# Exercise Sheet 04

**For this and future exercise sheets, you should treat all questions as potentially ones that you will need to think about for a while before you are able to solve them. Therefore, you are advised to read through each question after watching the associated lecture making appropriate notes. After watching all the lectures in the unit, you can return to attempt the questions in full.**


This markdown block is used to define macros for later markdown blocks. 
$\newcommand{\normaldist}{\mathcal{N}}$
$\newcommand{\betadist}{\text{Beta}}$
$\newcommand{\reals}{\mathbb{R}}$
$\newcommand{\ML}{\text{ML}}$
$\renewcommand{\vec}[1]{\boldsymbol{\mathbf{#1}}}$
$\newcommand{\matrix}[1]{\boldsymbol{\mathbf{#1}}}$
$\newcommand{\dataset}{\mathcal{D}}$
$\newcommand{\class}{\mathcal{C}}$
$\newcommand{\optimal}[1]{#1^{\star}}$
$\newcommand{\argmin}{\text{argmin}}$
$\newcommand{\argmax}{\text{argmax}}$
$\newcommand{\expct}{\mathbb{E}}$
$\newcommand{\entropy}[1]{H[#1]}$
$\newcommand{\conditionalentropy}[2]{\entropy{#1 | #2}}$
$\newcommand{\kldiv}[2]{KL(#1 || #2)}$
$\newcommand{\mutualinfo}[2]{I[#1;#2]}$
$\newcommand{\deriv}[2]{\frac{d}{d #2} \left( #1\right)}$
$\newcommand{\inputs}{\matrix{X}}$

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

import scipy.stats
import pandas as pd
from itertools import groupby

## Exercise 4.1 Univariate Gaussian estimation

In this question, you’ll generate some data from a 1d (one dimensional) Gaussian distribution, and then attempt to estimate the mean of that data. This will produce similar results to what you saw in the lectures. This is quite an extensive exercise, so you may choose to continue with the videos after a brief look, then return to it later. Note that **4.1 b)-d)** appear after the code block for **4.1 a)**.

### 💻 4.1 a) 
Look at the function `plot_simple_gaussian` below. This is called
by the main function, and should plot a Gaussian curve with the provided mean, mu , and
variance, sigma2 . However, you must correct the `prob_dens_gaussian` function in module `fomlads.data.function` to return the probability density function (PDF) of a Gaussian distribution. (It currently only returns an array of zeros in the same shape as the input xs .)

In [ ]:
## Exercise 4.1 a)
## find the imported function and correct it
from fomlads.data.function import prob_dens_gaussian

def plot_simple_gaussian(mu, sigma2, xlim=None):
    """
    This function plots a simple gaussian curve, with mean mu, variance sigma2
    in the x range xlim.
    """
    if xlim is None:
        sigma = np.sqrt(sigma2)
        xlim = (mu-3*sigma, mu+3*sigma)
    # get the x values we wish to plot at
    xs = np.linspace(xlim[0],xlim[1],101)
    # calculate the associated y values
    ys = prob_dens_gaussian(xs, mu, sigma2)
    # plot likelihood function
    fig = plt.figure()
    ax = fig.add_subplot(1,1,1)
    ax.plot(xs,ys)
    ax.set_xlabel("x")
    ax.set_ylabel("$p(x)$")
    fig.tight_layout()
    return fig, ax

plot_simple_gaussian(1, 3)

### 💻 4.1 b)
Now look at the `sample_and_plot_data_approx` function provided. You will edit this function, so that it draws the given number (`N`) of samples and plots these. You should plot the sampled points at their specified $x$ coordinate, with a $y$ coordinate of $0$. Uncomment the call to `sample_and_plot_data_approx` in `main` and run the script.

*You can generate samples from a Gaussian distribution with the `numpy.random.normal` library function (see [here](https://docs.scipy.org/doc/numpy-1.13.0/reference/generated/numpy.random.normal.html)). Note that this function takes standard deviation (not variance) as input.*

### 💻 4.1 c)

Look at the `max_lik_1d_gaussian` function in `fomlads.model.density_estimation`. Complete the function so that, treating the input `samples` as Gaussian samples, it returns maximum likelihood estimates of their mean and variance. Use this function within `sample_and_plot_data_approx` to plot the PDF of the maximum likelihood Gaussian probability density alongside the samples and the true distribution.

*The maximum likelihood estimates are given to you in the lecture slides at the end of the section on real variables. You can reuse `prob_dens_gaussian` to help plot the curve.*

### 💻 4.1 d) 

The maximum likelihood variance of data, $\sigma_{\ML}$, calculated in **4.1 c)**, typically under-estimates the true variance. This is most noticeable when the number of samples is small. Run your code again with just $3$ samples, is the estimated distribution narrower than the true distribution?

A better point estimate for the variance is:
$$\sigma^2 = \frac{N}{N-1}\sigma^2_{\ML} = \frac{1}{N-1}\sum_{n=1}^{N}(x_n -\mu_{\ML})^2$$
Include this estimate too on your plot.

*See Section 1.2.4 and Exercise 1.12 from (Bishop, 2006) for more detail on why $\sigma_{\ML}$ is an underestimate.*

In [ ]:
# you will need to locate this function and edit it
from fomlads.model.density_estimation import max_lik_1d_gaussian

## Exercises 4.1 b), d) and e) complete the following method
def sample_and_plot_data_approx(mu, sigma2, N):
    """
    Samples N points from a Gaussian with mean mu and variance sigma2, then
    plots these alongside the curve maximum likelihood estimate
    """ 
    # You should complete this function
    sigma = np.sqrt(sigma2)
    # example: to draw 10 samples from a gaussian of mean 0 and standard
    # deviation 1 use: np.random.normal(0,1,10)

mu = 0
sigma2 = 1
N = 2
sample_and_plot_data_approx(mu, sigma2, N)



## Exercise 4.2 Univariate Gaussian bayesian estimation

### 💻 4.2  a)
Consider a new function, `sample_and_plot_mean_approx`, which takes the following arguments: the number of samples, $N$; the true mean of the data, $\mu$; the true variance of the data, $\sigma^2$; a prior on the mean of $\mu$, $m_0$; and a prior on the variance of $\mu$, $s^2_0$. This should

* Sample $N$ datapoints from $\normaldist(.|\mu, \sigma^2)$.
* Find the maximum likelihood estimate for $\mu$ (we'll call it $\mu_\ML$).
* Assume the variance $\sigma^2$ is known, and find the Bayesian estimates, $m_N$ (mean) and $s^2_N$ (variance), on the posterior distribution of $\mu$ given the samples. You may want to write a function for this.
* Plot the samples (as points), $m_N$ and $\mu_\ML$ (as vertical lines) and the posterior $\normaldist(\mu|m_N, s^2_N)$ (as a curve) together on the same plot.

*Can you add a MAP estimate to the plot?*

### 💻 4.2 b)
Find the `sample_and_plot_data_approx` function in the code block above (for **4.1 b)-d)**). You should edit that function for this question. Within that function, sample $10$ means from the posterior $\normaldist(\mu|m_N, s^2_N)$ and use these mean samples to plot additional PDF curves on the same plot. This gives a set of distributions that are representative of the uncertainty in the Bayesian estimate.

You will need to provide values for prior parameters $m_0$ and $s^2_0$. What happens when $s^2_0$ is small and $m_0 \neq \mu$?

*Plot these new curves all in the same colour, and set the `linewidth` of the plots to a smaller value than your maximum likelihood curve, so that you can distinguish them on the plot.*

In [ ]:
### Complete the code parts of exercise 4.2 here

## Exercise 4.2 Solution:

*Complete any non-code parts to your solution in markdown below.*

## Exercise 4.3

### 📖 4.3 a)

Read Appendix C from (Bishop, 2006) if you haven't done so already. Then re-read Section 2.3. 

### ✏️ 4.3 b)

Attempt Exercise 2.17 from (Bishop, 2006), repeated here for convenience:

> 2.17 ( ) www Consider the multivariate Gaussian distribution given by (2.43). By
writing the precision matrix (inverse covariance matrix) Σ − 1 as the sum of a sym-
metric and an anti-symmetric matrix, show that the anti-symmetric term does not
appear in the exponent of the Gaussian, and hence that the precision matrix may be
taken to be symmetric without loss of generality. Because the inverse of a symmetric
matrix is also symmetric (see Exercise 2.22), it follows that the covariance matrix
may also be chosen to be symmetric without loss of generality.


## Exercise 4.3 b) solution

*complete the solution in markdown here*

## Exercise 4.4  Multivariate data

This question is intended for you to practice some data manipulation, as well as
gaining some insight into the 2-dimensional Gaussian distribution.

### 💻 4.4 a) 
Look at the function `sample_2d_isotropic_gaussian` in `fomlads.data.synthetic`. This is provided for you and generates `N` 2 dimensional data points, $(x_n, y_n)$, where each dimension is independently sampled from a different 1d Gaussian, i.e. $x_n\sim \normaldist(x_n|\mu_x, \sigma^2_x)$ and $y_n\sim \normaldist(y_n|\mu_y, \sigma^2_y)$ for all $n$.

Now look at the function `plot_2d_data_and_approximating_gaussian` in `fomlads.plot.exploratory`. When completed, `plot_2d_data_and_approximating_gaussian` will scatter plot this data, as well as a contour plot showing equiprobability surfaces (contours) of the Gaussian corresponding to the specified mean and covariance matrix. If the pair of variables is treated as a 2d vector random variable, what form do you expect the covariance matrix, `Sigma`, to take? And what shape do you expect the equiprobability surfaces to have?

*You do not need to edit any code at this point, just predict the answer to these two questions. See the lecture slide on **The Influence of $\Sigma$** to help you with this.*

### 💻 4.4 b)

Update the `plot_2d_data_and_approximating_gaussian` so that the two columns of the input parameter `data` are extracted and scatter plotted. You should also label the axes with the provided field-names.

Run the code, does this produce what you expected? Do not worry about the contours at this stage, you will deal with this in the next question.

*You may want to look back at the Old Faithful data visualisation question from Tutorial Sheet 1 for help with this question.*

### 💻 4.4 c)

In the provided code, the contours drawn by the script are always the same. This is because the function `max_lik_mv_gaussian` in `fomlads.model.density_estimation` is incomplete. To complete it, you must calculate the maximum likelihood mean and variance of your input `data`.

To do this, you must translate the mathematical expressions from the slide **Maximum likelihood estimates** into code. Assume you have data-matrix $\inputs \in \reals^{N\times D}$, whose $n$th row is data-point $\vec{x}_n^{T}$ sampled from a multivariate Gaussian. 

When `max_lik_mv_gaussian_approx` is correct, the contours drawn on the plot should correspond to the data. 

Run the data a number of times, and look at the resulting approximating Gaussian. As the data is isotropic, what shape would you expect for the contours? Is that always the case? Why/Why not?

*For $\vec{\mu}_{\ML}$ you may want to use the [`numpy.mean` function](https://docs.scipy.org/doc/numpy-1.13.0/reference/generated/numpy.mean.html). The covariance matrix involves calculating the outer product of vectors, e.g. $\vec{v}\vec{v}^T$ for some column vector $\vec{v}$ (giving a square matrix). Mostly you have been performing elementwise calculations up to this point but now we need some way of performing matrix multiplication. There are two concerns for this. First, you must make sure your matrices have the appropriate dimensions, and for vectors that means you need to decide whether you have a column vector (a $K\times 1$ array) or a row vector (a $1\times K$ array). For this, I suggest you read up on [`numpy.reshape`](https://numpy.org/doc/stable/reference/generated/numpy.reshape.html). Second, you have to ensure matrix multiplication rather than elementwise multiplication. There are a number of ways to do this. My preference is to use the `@` operator, but others prefer `numpy.matmul` -- see [here](https://blog.finxter.com/numpy-matmul-operator/) for a brief overview.*


In [ ]:
## Exercise 4.4 code
# provided function to import synthetic 2 dimensional gaussian data
from fomlads.data.synthetic import sample_2d_isotropic_gaussian
# Exercise 4.4 b) - you must edit this function
from fomlads.plot.exploratory import plot_2d_data_and_approximating_gaussian
# Exercise 4.4 c) - you must edit this function
from fomlads.model.density_estimation import max_lik_mv_gaussian

N = 100
data = sample_2d_isotropic_gaussian(N)
# TODO: complete the library function max_lik_mv_gaussian
mu, Sigma = max_lik_mv_gaussian(data)
# TODO: complete the library functions plot_2d_data_and_approximating_gaussian
plot_2d_data_and_approximating_gaussian(data, mu, Sigma, field_names=['x', 'y'])



## Exercise 4.5

### 💻 4.5 a) 

Download the `turtles.csv` file from the moodle course page. This data was originally published in (Jolicoeur, 1960), but the format has been adapted for ease of import. Read the data description provided alongside the downloaded files. This data describes a collection of turtles and includes gender and measurement attributes.

Look at the second code block below. Import the turtle data using `pandas.read_csv` and extract the heights and widths as a 2d numpy array with `shape = (N,2)`. Now plot the data alongside the equiprobability contours of the maximum likelihood Gaussian.  What does this tell you about the lengths and widths of turtles.


### 💻 4.5 b)

Consider the 3 dimensional data of lengths, widths and heights of the turtle data. Fit a maximum likelihood Gaussian to the 3 dimensional data. Now use the results from the slide **Partitioned Gaussian: result** to calculate

i. The conditional gaussian over widths and heights given that the length of the turtle is $150$cm. Plot this on a contour plot, with scattered data of all the widths and heights. Should the approximating distribution match the data?

ii. The marginal gaussian over lengths (marginalising out the widths and heights). Plot this as a line plot using `prob_dens_gaussian`, and overlay a histogram of all the lengths of the turtles. Should the approximating distribution match the data?

*You will need to perform more matrix multiplication here, so make sure you are familiar with how to do that. In particular, you will need to manage the dimensions of your variables (they should all be matrices even if they have only 1 element). You will also find the library function `numpy.linalg.inv` helpful for taking inverses.*

In [ ]:
## Exercise 4.5 a) - import the turtle data and plot as directed

## Exercise 4.5 b) i. - the conditional

## Exercise 4.5 ii. - the marginal


## Exercise 4.6 Reflection

### 💡 4.6 a)

There has been quite a bit of theory this week, so take some time to reflect on what you have seen. Think about what real world processes a Gaussian linear model might be used to approximate. Does it surprise you that all these conditional and marginal distributions turn out to also be Gaussian? Would it be true for any other distribution you might have encountered before, e.g. the Exponential.

### 💡 4.6 b)

Look back at the first exercise on this sheet. How would you approach the Bayesian approximation in **Exercise 4.2** if you did not know the variance of the data? Why might there be a problem mixing a Bayesian estimate of the mean, with a maximum-likelihood estimate of the variance? (Think about how $\sigma^2_{\ML}$ is calculated.) 

Read back through Sections 2.3-2.3.6 then on to the end of section 2.3 of (Bishop, 2006), then go back and complete any of the exercises you may have left.

*If you are interested in how to deal with this in a Bayesian way, then Section 2.3.6 in (Bishop, 2006) describes this in some detail.*


## Exercise 4.6 b) solution

*Use this block to write any notes on your reflections.*
